# Deep Fictitious Play for Stochastic Differential Games on Graphs

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import math
from math import isclose
import sys
import torch
from torch import nn

## Model Settings and Parameters

There are $N$ players in the game with the dynamics given by
$$dX_i(t) = \left[a\left(\frac{1}{d_{v_i}}\sum_{j:v_j\sim v_i}X_j(t) - X_i(t)\right) + \alpha^i_t\right]\,dt + \sigma\,dW_t^i$$
where $a$ is the mean-reverting parameter, $d_{v_i}$ is the degree of vertex $v_i$ in the graph, $\alpha^i_t$ is the control of player $i$ at time $t$ and $(W^0_t,...,W^{N-1}_t)$ is an $N$-dimensional Brownian motion. The $\sim$ relationship between two vertices $v_i,v_j$ in the graph means that there exists an edge connecting $v_i$ to $v_j$. The running cost of player $i$ in this problem is set as
$$f^i(X(t),\alpha^i_t) = \frac{1}{2}(\alpha^i_t)^2 - q\alpha^i_t\left(\frac{1}{d_{v_i}}\sum_{j:v_j\sim v_i}X_j(t) - X_i(t)\right) + \frac{\varepsilon}{2}\left(\frac{1}{d_{v_i}}\sum_{j:v_j\sim v_i}X_j(t) - X_i(t)\right)^2$$
the terminal cost of player $i$ is set as
$$g^i(X(t)) = \frac{c}{2}\left(\frac{1}{d_{v_i}}\sum_{j:v_j\sim v_i}X_j(t) - X_i(t)\right)^2$$
with $a,\sigma,q,\varepsilon,c>0$ as parameters. We want to derive the Markovian Nash equilibrium where each player minimizes its expected cost
$$J^i(\alpha) = \mathbb{E}\left[\int_0^Tf^i(X(t),\alpha^i_t)\,dt + g^i(X(T))\right]$$
with time horizon $[0,T]$.

In [3]:
# Model parameters (same as that in Ruimeng's paper)
N_PLAYER = 10
MOD_a = 1.0
MOD_sigma = 1.0
MOD_q = 0.0
MOD_eps = 1.0
MOD_c = 1.0
MOD_T = 1.0

The graph structure is encoded in the graph Laplacian $L$. Here we require the graph to be simple, connected and undirected, but not necessarily transitive. The graph Laplacian is defined as follows:
$$L\in\mathbb{R}^{N\times N},L_{ij} = \begin{cases}1 & i=j\\ -\frac{1}{\sqrt{d_{v_i}d_{v_j}}} & v_i\sim v_j\\ 0 & \text{else}\end{cases}$$

In [4]:
# Graph structure, LAP (N_PLAYER * N_PLAYER)
# For simplicity, set it as the complete graph for now and compare with the closed-form solution
LAP = -(1 / (N_PLAYER - 1)) * np.ones((N_PLAYER,N_PLAYER))
for loop_ind in range(N_PLAYER):
    LAP[loop_ind,loop_ind] = 1

Use feedforward NN to approximate the control of each player. The input is formed as time $t$ and state vector $x$ of all players at time $t$ so it has dimension $N + 1$, the output is a real number of dimension $1$.

In [5]:
class Model(nn.Module):
    # Standard feedforward NN
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential( 
            nn.Linear(N_PLAYER + 1,64),   # 3-layer, hidden layer of dimension 64
            nn.Sigmoid(),      
            nn.Linear(64, 64), 
            nn.Sigmoid(),      
            nn.Linear(64, 1),  
            )
    def forward(self,z):
        return self.net(z)

In [6]:
# Learning rate
LR_eta = 0.005

Since we need to numerically simulate the SDE solution with Euler scheme, it's good to specify how we are discretizing the time interval $[0,T]$ and how many samples we are using in Monte Carlo to approximate the expected cost.

In [7]:
# SDE simulation parameters
N_SIM = 2 ** 7 # Num of Monte Carlo samples (batch size)
N_STEPS = 25 # Num of time steps in time discretization

In fictitious play, specify the number of rounds we play until the convergence to NE. In each FP round, train the NN for the number of epochs specified below.

In [8]:
# FP parameter
N_EPOCH = 200 # Number of epochs in each player's training
N_FP_ROUND = 15 # Number of rounds of FP

## Graph Information

The main information we are using is the degree and neighbors of each vertex in the graph.

In [9]:
# Get the degrees and neighbors of each vertex in the graph
# Input: Graph Laplacian (N_PLAYER * N_PLAYER)
# Output: degree list (N_PLAYER), a list of list indicating neighbors
def get_graph_info(LAP):
    # Number of players
    N_PLAYER = LAP.shape[0]
    
    # Degree list and neighbor list (a list of list)
    deg_lst = list()
    nb_lst_lst = list()
    
    for vertex_i in range(N_PLAYER):
        LAP_col = LAP[:,vertex_i]
        nb_lst = list()
        deg_count = 0
        for vertex_j in range(N_PLAYER):
            # Do not count itself
            if vertex_i == vertex_j:
                continue
            if not isclose(LAP_col[vertex_j],0):
                # An edge detected
                deg_count = deg_count + 1
                nb_lst.append(vertex_j)
        # Record
        deg_lst.append(deg_count)
        nb_lst_lst.append(nb_lst)
    return deg_lst, nb_lst_lst

Get the degree list and the neighbor list that will be used later. Notice that the neighbor list is a list of list, its $i$-th entry is a list containing all neighbors of player $i$.

In [10]:
# Get the information of the graph (degree list, neighbor list)
DEG_LST, NB_LST_LST = get_graph_info(LAP)

# If there exists 0 degree, output error(graph assumed to be connected)
if 0 in DEG_LST:
    sys.exit(1)

## Dynamics

The following function computes the mean-reverting term $\left(\frac{1}{d_{v_i}}\sum_{j:v_j\sim v_i}X_j(t) - X_i(t)\right)$ for player $i$.

In [11]:
# Calculate mean-reverting term for a certain player
# Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of player
# Output: a 2-dim column tensor (N_SIM * 1)
def mean_rev_term(X_tensor,player_ind,NB_LST_LST):
    # All neighbors of that player
    nb_player_ind = NB_LST_LST[player_ind]
    
    # Maintain the dimension as 2-dim (N_SIM * 1)
    X_tensor_nb = X_tensor[nb_player_ind,:]
    mean_tensor = torch.unsqueeze(torch.mean(X_tensor_nb,dim = 0),1)
    player_ind_tensor = torch.unsqueeze(X_tensor[player_ind,:],1)
    return mean_tensor - player_ind_tensor

Build up functions in the dynamics of a certain player. Here it's taken as a linear-quadratic game.

In [12]:
# Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of the player, model parameters
# control of the player as 2-dim tensor (N_SIM * 1)
# Output: a 2-dim column tensor (N_SIM * 1)

# Drift coefficient
def dynamics_b(X_tensor,player_ind,NB_LST_LST,MOD_a,ctrl_player_ind):
    mean_rev = mean_rev_term(X_tensor,player_ind,NB_LST_LST)
    return MOD_a * mean_rev + ctrl_player_ind

# Running cost rate 
def dynamics_f(X_tensor,player_ind,NB_LST_LST,MOD_q,MOD_eps,ctrl_player_ind):
    mean_rev = mean_rev_term(X_tensor,player_ind,NB_LST_LST)
    return 0.5 * (ctrl_player_ind ** 2) - MOD_q * ctrl_player_ind * mean_rev + MOD_eps / 2.0 * (mean_rev ** 2)

# Terminal cost
def dynamics_g(X_tensor,player_ind,NB_LST_LST,MOD_c,ctrl_player_ind):
    mean_rev = mean_rev_term(X_tensor,player_ind,NB_LST_LST)
    return MOD_c / 2.0 * (mean_rev ** 2)

## NN Approximated Control

The following function calculates the NN-approximated control of all players by sending the current time and states into the NN.

In [13]:
# Get all players' controls at a fixed time step from NNs (at time num_dt * dt)
# Input: X_MC as an array of (N_SIM * 1) tensors, a list of NN
# Output: an array of 2-dim (N_SIM * 1) tensors denoting each player's control
def get_control_from_NN(NN_list,num_dt,dt,N_PLAYER,N_SIM,X_MC):
    ctrl = np.empty(N_PLAYER,'O') 
        
    # For each player, organize the same (t,x) pair to send into NN
    t_vec = num_dt * dt * torch.ones((N_SIM,1))
    NN_input = t_vec
    for player_ind in range(N_PLAYER):
        NN_input = torch.cat((NN_input,X_MC[player_ind]),1)
        
    # NN_input has each row of dimension (N + 1) with the
    # first entry as time and the remaining entries as states of all players
    # Output of the NN recorded as the control
    for player_ind in range(N_PLAYER):
        ctrl[player_ind] = NN_list[player_ind](NN_input)
    return ctrl

## An Auxiliary Function

The following function turns the array of 2-dimensional tensors of size $N_{SIM} \times 1$ into 2-dimensional tensors of size $N_{PLAYER} \times N_{SIM}$ to make vectorized calculations easier.

In [14]:
# Converting X_MC from an array (length N_PLPYER) of 2-dim (N_SIM * 1) tensors to a 2-dim (N_PLAYER * N_SIM) tensor
# Input: X_MC, model parameters
# Output: a 2-dim tensor (N_PLAYER * N_SIM)
def get_two_dim_tensor(X_MC,N_PLAYER,N_sim):
    X_MC_tensor = X_MC[0]
    # The state vector of all players
    for player_ind in range(1,N_PLAYER):
        X_MC_tensor = torch.cat((X_MC_tensor,X_MC[player_ind]),1)
    # Now it's N_sim * N_PLAYER shape, do transpose to get N_PLAYER * N_SIM shape
    return torch.transpose(X_MC_tensor,0,1)

## SDE Simulation and Loss Calculation

The following functions are used in the simulation of the solution to the SDE of all players with the control of all players provided. However, only player $i$'s cost will be recorded in the loss function and its control will be updated. The loss function is just formed as the squared expected total loss of player $i$.

The Euler scheme is given by:
$$X_i(t_{p+1}) - X_i(t_p) = \left[a\left(\frac{1}{d_{v_i}}\sum_{j:v_j\sim v_i}X_j(t_p) - X_i(t_p)\right) + \alpha^i_{t_p}\right]\,(t_{p+1} - t_p) + \sigma\,(W^i_{t_{p+1}} - W^i_{t_p})$$
for the initial condition $X_i(0)$ of each player, we set it through the function below.

In [15]:
# Set initial value condition for the state dynamics
# Input: X_MC as an array of 2-dim tensors (N_SIM * 1), model parameters
# Output: X_MC updated with initial value conditions
def set_init_val(X_MC,N_PLAYER,N_SIM):
    init_X_MC = X_MC
    ## All zero initial value
    #for player_ind in range(N_PLAYER):
        #init_X_MC[player_ind] = torch.zeros((N_SIM,1))
        
    ## Condition in Ruimeng's paper when there are 5 players (verified to work)
    #if N_PLAYER == 5:
        #init_X_MC[0] = 1 * torch.ones((N_SIM,1))
        #init_X_MC[1] = 5 * torch.ones((N_SIM,1))
        #init_X_MC[2] = 7 * torch.ones((N_SIM,1))
        #init_X_MC[3] = 3 * torch.ones((N_SIM,1))
        #init_X_MC[4] = 8 * torch.ones((N_SIM,1))
        
    # Generally, we can set x0 = 0.5 * i for player i
    for player_ind in range(N_PLAYER):
        init_X_MC[player_ind] = 0.5 * player_ind * torch.ones((N_SIM,1))
    return init_X_MC

In [16]:
# Simulate SDE with Euler scheme and compute the loss for a certain player (referred to as fixed player)
# Input: NN list and optimizer list, index of the player, model parameters
# Output: a real number as the loss of the player
def simulate_SDE_FP(NN_list,opt_list,player_fixed_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS,N_SIM):
    # Calculate dt
    dt = MOD_T / N_STEPS
    
    # Let X be an array of a torch tensors of size N_SIM so we can do the Monte Carlo once and for all
    # All players' states at a fixed time step
    X_MC = np.empty(N_PLAYER,'O')
    next_X_MC = np.empty(N_PLAYER,'O')
    
    # Record the cumulative running cost for the certain player
    cumu_running_cost_MC = torch.zeros((N_SIM,1))
    
    # Initial condition set for all players
    X_MC = set_init_val(X_MC,N_PLAYER,N_SIM)
    #-------------------------------------------------------------
    # At each time step
    for num_dt in range(N_STEPS):
        
        # Get the control of each player from NN (an array of 2-dim tensors (N_SIM * 1))
        ctrl = get_control_from_NN(NN_list,num_dt,dt,N_PLAYER,N_SIM,X_MC)
        
        # Convert X_MC to a 2-dimensional tensor (N_PLAYER * N_SIM)
        X_MC_tensor = get_two_dim_tensor(X_MC,N_PLAYER,N_SIM)
        
        # Calculate the state of all players at the next time step
        for player_loop_ind in range(N_PLAYER):
            
            # Get this player's control (2-dim tensor (N_SIM * 1))
            ctrl_player_loop_ind = ctrl[player_loop_ind]
            
            # Set up the drift term for this player in Euler scheme (2-dim column tensor)
            drift = dynamics_b(X_MC_tensor,player_loop_ind,NB_LST_LST,MOD_a,ctrl_player_loop_ind)
            
            # BM trajectory
            dW_t = torch.randn((N_SIM,1)) * np.sqrt(dt)
            
            # Vectorized calculations for MC samples, Euler scheme
            next_X_MC[player_loop_ind] = X_MC[player_loop_ind] + drift * dt + MOD_sigma * dW_t
        
        # Cumulative running cost for that fixed player
        # Notice that we shall multiply by dt since it's an integral
        ctrl_player_fixed_ind = ctrl[player_fixed_ind]
        cumu_running_cost_MC = cumu_running_cost_MC + \
                        dt * dynamics_f(X_MC_tensor,player_fixed_ind,NB_LST_LST,MOD_q,MOD_eps,ctrl_player_fixed_ind)
        
        # Proceed to the next time step
        X_MC = next_X_MC
            
    # Terminal cost for that fixed player
    ctrl_player_fixed_ind = ctrl[player_fixed_ind]
    terminal_cost_MC = dynamics_g(X_MC_tensor,player_fixed_ind,NB_LST_LST,MOD_c,ctrl_player_fixed_ind)
    
    # Calculate loss of that fixed player as expected cost (approximated by Monte Carlo)
    loss_fixed_player = torch.mean((cumu_running_cost_MC + terminal_cost_MC))
    
    return loss_fixed_player

## Training Loop

Create a list of NNs, one for each player. Since we are doing deep fictitious play, we have to fix all other players' controls and train only a single player's control so we shall maintain the parameters and optimizers separately.

In [17]:
# Initialize NNs to approximate the control of each player
NN_list = list()
opt_list = list()
for _ in range(N_PLAYER):
    NN = Model()
    opt = torch.optim.Adam(NN.parameters(),lr = LR_eta)
    NN_list.append(NN)
    opt_list.append(opt)

Alternating deep fictitious play, use the latest strategy immediately after it's trained. Expect to see the control of all players converge to the NE.

In [ ]:
# Record the loss change for player 0 w.r.t. FP round
player_0_loss_lst = list()

# In each round of FP
for FP_ind in range(N_FP_ROUND):
    # Fix all other players' control and only update this single player's control
    for player_ind in range(N_PLAYER):
        
        # Train this single player's control for a number of epochs (single-batch)
        for epoch_ind in range(N_EPOCH):
        
            # Simulate SDE with Monte Carlo and compute loss of the certain player
            loss = simulate_SDE_FP(NN_list,opt_list,player_ind,MOD_T,
                               NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS,N_SIM)
        
            # Print loss every 5 FP rounds for the first player
            if player_ind == 0 and epoch_ind == 0:
                print('FP round: ' + str(FP_ind) + ', loss: ' + str(loss))
        
            # Update control of that single player in FP
            loss.backward()
            opt_list[player_ind].step()
        
            # Clear the gradient records for all NNs
            for any_player in range(N_PLAYER):
                opt_list[any_player].zero_grad()
        
        # Record the training loss of player 0
        if player_ind == 0:
            player_0_loss_lst.append(loss.detach().item())

FP round: 0, loss: tensor(2.0205, grad_fn=<MeanBackward0>)
FP round: 1, loss: tensor(1.5833, grad_fn=<MeanBackward0>)
FP round: 2, loss: tensor(1.5781, grad_fn=<MeanBackward0>)


In [ ]:
# Loss curve for player 0
plt.plot(range(N_FP_ROUND),player_0_loss_lst)
plt.title('Training Loss Curve')
plt.xlabel('FP Round')
plt.ylabel('Expected Cost of Player 0')

## Compare with the Closed-form Solution on Complete Graph

The NE $\hat{\alpha}(t,x)$ for such model (on observing current time $t$ and current state vector $x$) on complete graph actually has closed-form solution as provided below.
$$\hat{\alpha}(t,x) = (q + \eta_t)(\overline{x}_{(-i)} - x_i)$$
where $\overline{x}_{(-i)} = \frac{1}{N-1}\sum_{j=1,j\neq i}^N x_j$ is the mean state of all players except player $i$ and $\eta_t$ is a deterministic function in $t$.

In [ ]:
# Mean reverting term with the mean not including the certain player itself
# Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of player
# Output: a 2-dim column tensor (N_SIM * 1)
def mean_rev_term_all_except_one(X_tensor,player_fixed_ind,N_PLAYER):
    # Get all players' indices except the given one
    except_one_player_ind = list(range(player_fixed_ind)) + list(range(player_fixed_ind + 1,N_PLAYER))
    
    # Maintain the dimension as 2-dim (N_SIM * 1)
    X_tensor_except_one = X_tensor[except_one_player_ind,:]
    mean_tensor = torch.unsqueeze(torch.mean(X_tensor_except_one,dim = 0),1)
    player_fixed_ind_tensor = torch.unsqueeze(X_tensor[player_fixed_ind,:],1)
    return mean_tensor - player_fixed_ind_tensor

The formula of $\eta_t$ is given below.
$$\begin{cases}
        \eta_t = \frac{(c\delta^- - C) -(c\delta^+ - C)e^{(\delta^+ - \delta^-)(T-t)}}{(Ac-\delta^+) + (\delta^- - Ac)e^{(\delta^+ - \delta^-)(T-t)}}\\
        A = \frac{N+1}{N-1}\\
        B = \frac{2N}{N-1}(a+q)\\
        C = -(\varepsilon-q^2)\\
        R = \frac{B^2}{4} -AC\\
        \delta^\pm = -\frac{B}{2}\pm \sqrt{R}
    \end{cases}$$

In [ ]:
# Compute eta(t)
# Input: time t as a real number and model parameters
# Output: eta(t) as a real number
def function_eta(t,N_PLAYER,MOD_a,MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T):
    A = (N_PLAYER + 1) / (N_PLAYER - 1)
    B = 2 * N_PLAYER / (N_PLAYER - 1) * (MOD_a + MOD_q)
    C = -(MOD_eps - MOD_q ** 2)
    R = (B ** 2 / 4) - A * C
    delta_plus = -(B / 2) + np.sqrt(R)
    delta_minus = -(B / 2) - np.sqrt(R)
    exp_factor = np.exp((delta_plus - delta_minus) * (MOD_T - t))
    numer = (MOD_c * delta_minus - C) - (MOD_c * delta_plus - C) * exp_factor
    denom = (A * MOD_c - delta_plus) + (delta_minus - A * MOD_c) * exp_factor
    return numer / denom

In [ ]:
# Get all players' controls at a fixed time step from the true NE
# Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM)
# Output: an array of 2-dim (N_SIM * 1) tensors denoting each player's control
def get_true_NE(X_tensor,num_dt,dt,N_PLAYER,N_SIM,MOD_a,MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T):
    ctrl = np.empty(N_PLAYER,'O') 
    # Current time
    t = num_dt * dt
    
    # Under current time, calculate eta(t)
    eta_value = function_eta(t,N_PLAYER,MOD_a,MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T)
    for player_ind in range(N_PLAYER):
        ctrl[player_ind] = (MOD_q + eta_value) * mean_rev_term_all_except_one(X_tensor,player_ind,N_PLAYER)
    return ctrl

## The Test Function

Build up a test function to compare the expected cost under DFP solution and the closed-form solution. 

The idea is to simulate SDE under NN approximated and closed-form solution controls with the same BM trajectories.

In [ ]:
# Compute the expected cost based on given controls of all players (NN-approximated or closed-form)
# Input: NNs, model parameters, get_ctrl and get_ctrl_flag indicating how we are getting our controls
# get_ctrl_flag can only take values 'NN' or 'closed-form'.
# The last parameter N_SIM denotes the number of Monte Carlo iterations in the testing process, which is typically 
# much more than the batch size (although they share the same name), by default we take it as 10^5.
# Output: the expected cost as a real number
def get_exp_cost(NN_list,opt_list,player_fixed_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS,
                    get_ctrl,get_ctrl_flag,N_SIM_TEST = 10 ** 5):
    # Calculate dt
    dt = MOD_T / N_STEPS
    
    # Let X be an array of a torch tensors of size N_SIM so we can do the Monte Carlo once and for all
    # All players' states at a fixed time step
    X_MC = np.empty(N_PLAYER,'O')
    next_X_MC = np.empty(N_PLAYER,'O')
    
    # Record the cumulative running cost for the certain player
    cumu_running_cost_MC = torch.zeros((N_SIM_TEST,1))
    
    # Initial condition set for all players
    X_MC = set_init_val(X_MC,N_PLAYER,N_SIM_TEST)
    #-------------------------------------------------------------
    # At each time step
    for num_dt in range(N_STEPS):
        
        # Convert X_MC to a 2-dimensional tensor (N_PLAYER * N_SIM)
        X_MC_tensor = get_two_dim_tensor(X_MC,N_PLAYER,N_SIM_TEST)
        
        # Get the control of each player from NN (an array of 2-dim tensors (N_SIM * 1))
        # Here we have to check the get_ctrl_flag and call the get_ctrl function
        if get_ctrl_flag == 'NN':
            ctrl = get_ctrl(NN_list,num_dt,dt,N_PLAYER,N_SIM_TEST,X_MC)
        elif get_ctrl_flag == 'closed-form':
            ctrl = get_ctrl(X_MC_tensor,num_dt,dt,N_PLAYER,N_SIM_TEST,MOD_a,
                                MOD_sigma,MOD_q,MOD_eps,MOD_c,MOD_T)
        else:
            sys.exit(1)
        
        # Calculate the state of all players at the next time step
        for player_loop_ind in range(N_PLAYER):
            
            # Get this player's control (2-dim tensor (N_SIM * 1))
            ctrl_player_loop_ind = ctrl[player_loop_ind]
            
            # Set up the drift term for this player in Euler scheme (2-dim column tensor)
            drift = dynamics_b(X_MC_tensor,player_loop_ind,NB_LST_LST,MOD_a,ctrl_player_loop_ind)
            
            # BM increments
            dW_t = torch.randn((N_SIM_TEST,1)) * np.sqrt(dt)
            
            # Vectorized calculations for MC samples, Euler scheme
            next_X_MC[player_loop_ind] = X_MC[player_loop_ind] + drift * dt + MOD_sigma * dW_t
        
        # Cumulative running cost for that certain player
        # Notice that we shall multiply by dt since it's an integral
        ctrl_player_fixed_ind = ctrl[player_fixed_ind]
        cumu_running_cost_MC = cumu_running_cost_MC + dt * dynamics_f(X_MC_tensor,
                                        player_fixed_ind,NB_LST_LST,MOD_q,MOD_eps,ctrl_player_fixed_ind)
        
        # Proceed to the next time step
        X_MC = next_X_MC
            
    # Terminal cost for that certain player
    ctrl_player_fixed_ind = ctrl[player_fixed_ind]
    terminal_cost_MC = dynamics_g(X_MC_tensor,player_fixed_ind,NB_LST_LST,MOD_c,ctrl_player_fixed_ind)
    
    # Calculate expected total cost
    exp_cost = torch.mean((cumu_running_cost_MC + terminal_cost_MC)).detach().item()
    
    return exp_cost

Expected cost of the closed-form solution.

In [ ]:
# Get expected cost of all players (closed-form solution)
get_ctrl_flag = 'closed-form'
get_ctrl = get_true_NE
for player_fixed_ind in range(N_PLAYER):
    true_exp_cost = get_exp_cost(NN_list,opt_list,player_fixed_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS,
                    get_ctrl,get_ctrl_flag)
    print('The expected cost for player ' + str(player_fixed_ind) + ' is: ' + str(true_exp_cost))

Expected cost of the NN approximated solution.

In [ ]:
# Get expected cost of all players (NN approximated)
get_ctrl_flag = 'NN'
get_ctrl = get_control_from_NN
for player_fixed_ind in range(N_PLAYER):
    NN_exp_cost = get_exp_cost(NN_list,opt_list,player_fixed_ind,
                    MOD_T,NB_LST_LST,MOD_a,MOD_q,MOD_eps,MOD_c,MOD_sigma,N_PLAYER,N_STEPS,
                    get_ctrl,get_ctrl_flag)
    print('In DFP, the expected cost for player ' + str(player_fixed_ind) + ' is: ' + str(NN_exp_cost))